In [8]:
import pandas as pd
import numpy as np

from pathlib import Path

In [9]:
pd.options.display.max_rows = 100
pd.options.display.min_rows = 20
pd.options.display.max_columns = None

In [10]:
ROOT_PATH = Path(Path.cwd()).resolve().parent
DATA_PATH = ROOT_PATH/'data'/'raw'/'acb'
PROCESSED_PATH = ROOT_PATH/'data'/'processed'

# Creación de DataFrame principal

In [11]:
dfs = []

for file in Path(DATA_PATH).glob('*.csv'):
    jornada = int(file.stem.split(' - ')[-1])
    df_temp = pd.read_csv(file)
    df_temp['Jornada'] = jornada
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)
df = df.sort_values(by=['D', 'Jornada']).reset_index(drop=True)

## Cambio de nombre de columnas

In [12]:
df = df.rename(columns= {'#ERROR!': '+/-'})

# Data engienieering

## Titular 

In [13]:
# Creamos la columna Titular basándonos en la marca * junto al dorsal (D) que lo indica
df['Titular'] = df['D'].str.contains(r'\*', na=False)
# df['Titular'] = np.where(df['Titular'] == True, 1, 0)

## Dorsal

In [14]:
# Corregimos la columna D
df['D'] = df['D'].str.replace('*', '', regex=False)
df['D'] = df['D'].astype(str).str.zfill(2)
df['D'] = df['D'].str.replace('nan', 'UCAM', regex=False)

# Se elimina la columna nombre, que no aporta información útil
df.drop(columns='Nombre', inplace=True)

# Se renombra la columna Dorsal como #
df = df.rename(columns={'D':'#'})

## Tiros

In [15]:
# Separamos los tiros en anotados y totales. También creamos una categoría para los fallados. El porcentaje de acierto lo calcularemos directamente en PowerBI

# Tiros de 2
df[['T2A', 'T2I']] = (
    df['T2']
    .str.split('/', expand=True)
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
    .astype(int)
)
df['T2F'] = df['T2I'] - df['T2A']

# Triples
df[['T3A', 'T3I']] = (
    df['T3']
    .str.split('/', expand=True)
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
    .astype(int)
)
df['T3F'] = df['T3I'] - df['T3A']

# Tiros de Campo
df['TCA'] = df['T2A'] + df['T3A']
df['TCI'] = df['T2I'] + df['T3I']
df['TCF'] = df['T2F'] + df['T3F']


# Tiros libres
df[['TLA', 'TLI']] = (
    df['T1']
    .str.split('/', expand=True)
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
    .astype(int)
)
df['TLF'] = df['TLI'] - df['TLA']

# Mates
df['MAT'] = df['MAT'].fillna(0).astype(int)


# Eliminamos columnas innecesarias
cols_eliminar = [
    'T2', 'T2 %',
    'T3', 'T3 %',
    'T1', 'T1 %'
]
df = df.drop(columns=cols_eliminar)

## Rebotes

In [16]:
df[['RD', 'RO']] = (
    df['REBD+O']
    .str.split('+', expand=True)
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
    .astype(int)
)

df = df.rename(columns={'REBT':'REB'})
df['REB'] = df['REB'].fillna(0).astype(int)


In [17]:
# Combrobamos que la suma de ofensivos y defensivos nos da los totales en todos los casos
check_rebotes = (df['REB'].fillna(0) == df['RD'] + df['RO']).all()

if check_rebotes:
    print('✅Comprobación con ÉXITO')
else:
    print('⚠️⚠️ALERTA⚠️⚠️. La comprobación ha detectado un error')

✅Comprobación con ÉXITO


In [18]:
# Eliminamos columnas innecesarias
df = df.drop(columns='REBD+O')

## Asistencias

In [19]:
df['ASI'] = df['ASI'].fillna(0).astype(int)

## Robos y Pérdidas

In [20]:
df = df.rename(columns={'BR':'ROB', 'BP':'PER'})
df['ROB'] = df['ROB'].fillna(0).astype(int)
df['PER'] = df['PER'].fillna(0).astype(int)

## Puntos en Contraataque

In [21]:
df = df.rename(columns={'CONTRA':'PTSCATQ'})
df['PTSCATQ'] = df['PTSCATQ'].fillna(0).astype(int)

## Tapones

In [22]:
df = df.rename(columns={'TAPF':'TAP', 'TAPC':'TAPR'})
df['TAP'] = df['TAP'].fillna(0).astype(int)
df['TAPR'] = df['TAPR'].fillna(0).astype(int)

## Faltas

In [23]:
df = df.rename(columns={'FPF':'FP', 'FPC':'FPR'})
df['FP'] = df['FP'].fillna(0).astype(int)
df['FPR'] = df['FPR'].fillna(0).astype(int)

## +/- y Valoración

In [24]:
df['+/-'] = df['+/-'].fillna(0).astype(int)
df['VAL'] = df['VAL'].fillna(0).astype(int)

## Puntos

In [25]:
df = df.rename(columns={'P':'PTS'})
df['PTS'] = df['PTS'].fillna(0).astype(int)

## Minutos

In [26]:
# Se cambia la información del tiempo jugado a segundos en SEG y se elimina MIN
df['SEG'] = (
    df['Min']
    .fillna('00:00')
    .str.split(':')
    .apply(lambda x: int(x[0]) * 60 + int(x[1]))
)

df.drop(columns='Min', inplace=True)

# Visualización

In [27]:
pd.options.display.max_rows = None

df = df.sort_values(by=['#', 'Jornada']).reset_index(drop=True)
df

,#,PTS,REB,ASI,ROB,PER,PTSCATQ,TAP,TAPR,MAT,FP,FPR,+/-,VAL,Jornada,Titular,T2A,T2I,T2F,T3A,T3I,T3F,TCA,TCI,TCF,TLA,TLI,TLF,RD,RO,SEG
0,00,8,8,1,2,1,0,0,0,2,3,2,8,14,1,False,3,6,3,0,0,0,3,6,3,2,2,0,5,3,1001
1,00,11,7,0,0,1,2,0,0,1,4,7,25,16,2,False,3,6,3,0,0,0,3,6,3,5,6,1,4,3,1303
2,00,10,3,1,1,3,2,0,0,0,3,2,-1,11,3,True,3,3,0,0,0,0,3,3,0,4,4,0,2,1,1246
3,00,4,0,1,0,2,0,0,0,1,4,1,-9,0,4,True,2,2,0,0,0,0,2,2,0,0,0,0,0,0,815
4,00,20,6,0,1,4,0,0,1,3,5,5,3,20,5,True,9,10,1,0,0,0,9,10,1,2,3,1,4,2,1282
5,00,5,4,0,0,1,0,0,0,1,2,4,-5,7,6,True,2,4,2,0,0,0,2,4,2,1,2,1,2,2,656
6,00,2,3,0,1,1,0,0,1,0,2,4,-6,4,7,True,1,3,2,0,0,0,1,3,2,0,0,0,0,3,772
7,00,7,3,2,0,1,0,1,0,0,3,3,19,8,8,True,3,4,1,0,0,0,3,4,1,1,4,3,2,1,741
8,00,19,8,3,1,1,0,0,0,2,3,3,29,25,9,True,9,11,2,0,0,0,9,11,2,1,4,3,6,2,1286
9,00,9,3,0,2,2,2,1,0,1,4,4,8,8,11,False,3,6,3,0,0,0,3,6,3,3,5,2,3,0,1267


# Export

In [28]:
df.to_csv(
    PROCESSED_PATH/'Datos_acb.csv',
    sep=';',
    decimal=',',
    index=False
)